<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/forest_area_class_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  Forest Threshold & Type Analysis: Case Study
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [ ]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────


In [1]:
# ── CELL 2: conect to drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
print('✅ connected to drive')


Mounted at /content/drive
✅ connected to drive


In [2]:
# ── CELL 3: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main


print('✅ Ready')

Enter PAT: ··········
/content/Biochar_forest_estimation
HEAD is now at 2d3e790 task for are GLC countries and states, forest area countries  bin per type
✅ Ready


In [ ]:
import ee

# ── CELL 4: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')

treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

In [4]:
# ── CELL 5: import libraries and data from data_config ─────────────────────
import shutil
import geemap
import pandas as pd
import time
import matplotlib.pyplot as plt
import numpy as np
from data_config import (FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup,
                         forest_bins, country_thresholds, state_thresholds, forestClasses
                        )


from gee_functions import (
    # Section 1: Forest area by threshold — countries
    prepare_forest_collection,
    export_forest_area,
    # Section 2: Forest area by threshold — states
    prepare_states_forest_collection,
    export_states_forest_area,
    # Section 3: Forest cover bins — countries
    get_forest_cover_bins_one_country,
    export_forest_cover_bins_all_countries,
    # Section 4: Forest cover bins — states
    get_forest_cover_bins_one_state,
    export_forest_cover_bins_all_states,
    # Section 5: GLC forest type area — countries
    get_forest_cover_area_type_country,
    export_forest_cover_area_type_all_countries,
    # Section 6: GLC forest type area — states
    get_forest_cover_area_type_state,
    export_forest_cover_area_type_all_states,
    #section 7 : area for each bin for each GLC forest type
    get_forest_area_bin_type_state,
    export_forest_area_bin_type_all_states,
    export_forest_area_bin_type_all_countries, # Added this line
    export_glc_total_forest_area_all_countries,
    export_glc_total_forest_area_all_states,
    copy_gee_exports_to_repo,
    compute_forest_type_composition,
  )

import gee_functions
gee_functions.treecover2000_masked = treecover2000_masked
gee_functions.datamask             = datamask
gee_functions.glc_2000             = glc_2000
gee_functions.glc_2000_forest      = glc_2000_forest

print('✅ GEE objects injected into gee_functions')
print('✅ All GEE functions imported')
print('✅ Libraries imported')
print('✅ Data config loaded')

NameError: name 'treecover2000_masked' is not defined

In [ ]:
# ── CELL 6: Run Exports — United States All States ────────────────────────────

#print('── Submitting forest area by threshold task ──')
#state_threshold_task = export_states_forest_area(us_state_names, state_thresholds)

#print('\n── Submitting GLC forest type area task ──')
#state_glc_task = export_forest_cover_area_type_all_states(us_state_names, forestClasses)
#print('\n── Submitting forest cover bins task ──')
#state_bins_task = export_forest_cover_bins_all_states(us_state_names, forest_bins)

# Oregon only
#export_forest_area_bin_type_all_states(['Oregon'], forest_bins, forestClasses, region_label='Oregon')

# All states
#export_forest_area_bin_type_all_states(us_state_names, forest_bins, forestClasses, region_label='all_states')

# Pacific Northwest states
#export_forest_area_bin_type_all_states(['Oregon', 'Washington', 'California'], forest_bins, forestClasses, region_label='pacific_northwest')

In [ ]:
glc_states_task = export_glc_total_forest_area_all_states(
    us_state_names, region_label='all_states')

In [ ]:
print('\n── Submitting GLC total forest area — countries ──')
glc_countries_task = export_glc_total_forest_area_all_countries(
    FAO_LSIB_REGION,
    region_label='all_countries'
)

In [ ]:

print('── Submitting forest cover bins — all countries ──')
country_bins_task = export_forest_cover_bins_all_countries(
    FAO_LSIB_REGION,
    forest_bins,
    region_label='all_countries'
)

print('\n── Submitting GLC forest type area — all countries ──')
country_glc_task = export_forest_cover_area_type_all_countries(
    FAO_LSIB_REGION,
    forestClasses,
    region_label='all_countries'
)

print('\n── Submitting forest area bin type — all countries ──')
country_bin_type_task = export_forest_area_bin_type_all_countries(
    FAO_LSIB_REGION,
    forest_bins,
    forestClasses,
    region_label='all_countries'
)

In [ ]:
# ── CELL 7: Load, clean and save each file ───────────────────────────────────
GEE_FOLDER  = '/content/drive/MyDrive/GEE_exports/'
DATA_FOLDER = '/content/Biochar_forest_estimation/data/'
os.makedirs(DATA_FOLDER, exist_ok=True)



copy_gee_exports_to_repo(filenames = ['forest_area_bin_type_all_states.csv'], gee_folder= GEE_FOLDER,  data_folder= DATA_FOLDER)

forest_area_bin_type_all_states = pd.read_csv(DATA_FOLDER + 'forest_area_bin_type_all_states.csv')


In [ ]:
forest_composition_usa, forest_stats_usa = compute_forest_type_composition(
    forest_area_bin_type_all_states,
    excluded_bins=['10-20']
)

print(forest_stats_usa)
print(forest_composition_usa)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

forest_stats_sorted = forest_composition_usa.sort_values('weighted_mean_canopy')

# shorten names — remove class code number
forest_stats_sorted['short_name'] = forest_stats_sorted['glc_class'].str[3:]

ax.barh(
    forest_stats_sorted['short_name'],
    forest_stats_sorted['weighted_mean_canopy'],
    xerr=forest_stats_sorted['weighted_std_canopy'],
    color='#4C9BE8',
    edgecolor='white',
    capsize=4,
    alpha=0.8
)

ax.axvline(20, color='black', linestyle='--', linewidth=1, label='US threshold (20%)')
ax.set_xlabel('Weighted mean canopy cover (%)')
ax.set_title('Typical canopy cover by GLC forest type — US states')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/content/Biochar_forest_estimation/figures/forest_type_canopy_cover_usa_all_.states.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import geemap

# apply Hansen 20% threshold mask
hansen_mask      = treecover2000_masked.gte(20)
glc_hansen       = glc_2000_forest.updateMask(hansen_mask)
treecover_masked = treecover2000_masked.updateMask(hansen_mask)

# visualization parameters
bin_vis = {
    'min': 20,
    'max': 100,
    'palette': ['#d9f0a3', '#addd8e', '#78c679', '#41ab5d', '#238443', '#006837', '#004529']
}

glc_vis = {
    'min': 51,
    'max': 92,
    'palette': [
        '4c7300', '006400', 'a8c800', '00a000',
        '005000', '003c00', '286400', '285000',
        'a0b432', '788200'
    ]
}

# split map
left_layer  = geemap.ee_tile_layer(treecover_masked, bin_vis,  'Hansen canopy cover')
right_layer = geemap.ee_tile_layer(glc_hansen,       glc_vis,  'GLC forest types')

Map = geemap.Map()
Map.setCenter(-98, 40, 4)
Map.split_map(left_layer, right_layer)
Map

In [5]:
GEE_FOLDER  = '/content/drive/MyDrive/GEE_exports/'
DATA_FOLDER = '/content/Biochar_forest_estimation/data/'
copy_gee_exports_to_repo(filenames=[
        'forest_area_bin_type_all_countries.csv',
        'glc_total_forest_area_all_countries.csv',
        'forest_cover_area_type_all_countries.csv',
    ], gee_folder= GEE_FOLDER,  data_folder= DATA_FOLDER)

✅ Copied and cleaned forest_area_bin_type_all_countries.csv
✅ Copied and cleaned glc_total_forest_area_all_countries.csv
✅ Copied and cleaned forest_cover_area_type_all_countries.csv


In [7]:
forest_area_bin_type_all_countries= pd.read_csv(DATA_FOLDER + 'forest_area_bin_type_all_countries.csv')
glc_total_forest_area_all_countries = pd.read_csv(DATA_FOLDER + 'glc_total_forest_area_all_countries.csv')
forest_cover_area_type_all_countries = pd.read_csv(DATA_FOLDER + 'forest_cover_area_type_all_countries.csv')
print(f'forest_area_bin_type_all_countries: {forest_area_bin_type_all_countries.head()}')
print(f'glc_total_forest_area_all_countries: {glc_total_forest_area_all_countries.head()}')
print(f'forest_cover_area_type_all: {forest_cover_area_type_all_countries.head()}')


forest_area_bin_type_all_countries:     country  51 Open evergreen broadleaved - 10-20  \
0      Chad                               0.008492   
1    Malawi                               0.042171   
2    Zambia                               0.079971   
3  Zimbabwe                               0.086092   
4  Botswana                               0.004359   

   51 Open evergreen broadleaved - 20-30  \
0                               0.001310   
1                               0.015338   
2                               0.053024   
3                               0.010647   
4                               0.000142   

   51 Open evergreen broadleaved - 30-40  \
0                               0.000095   
1                               0.005040   
2                               0.011463   
3                               0.001738   
4                               0.000004   

   51 Open evergreen broadleaved - 40-50  \
0                               0.000025   
1                    

In [6]:
# ── CELL 12: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in globals():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)

/content/Biochar_forest_estimation
[main ff10fb5] update results
 5 files changed, 663 insertions(+)
 create mode 100644 data/forest_area_bin_type_all_countries.csv
 create mode 100644 data/forest_cover_area_type_all_countries.csv
 create mode 100644 data/glc_total_forest_area_all_countries.csv
✅ Pushed to GitHub
